# Battle 通用远程 GPU Worker（单 cell 极简版）

**用法**
- **Push（推荐）**：`mode="push"`，填 `hub_token` + `push_token`（`registry_url` 留空则用 `hub_url`）。
  Run → 先取代码（hub `/code` → GitHub clone）→ 起服务 → 自动注册 → hub 按能力键派发。
  默认**不再** push-first（那会死锁在「等 code 的 job」）；`allow_push_first` 只作最后兜底。
- **Pull**：`mode="pull"` + `hub_url`/`hub_token`（行为与旧版逐字节相同）。

| 步骤 | 住哪 |
|---|---|
| CFG / 保活 / 薄分支 | 本 cell |
| Push 代码来源（`/code` 有界等待 → GitHub clone → push-first 兜底） | 本 cell（首包前无 code.zip） |
| 自动注册 / URL 换名重注册 / 注销 | `remote/push_register.py` + `remote/notebook_runtime.py` |
| 设备探测 / 守候日志 / 监督 / cloudflared 完整逻辑 | `remote/notebook_runtime.py` + `remote/push_bootstrap.py`（code.zip） |

修运行时 → 改仓库后重跑 cell / 换 job 即可，**不必重发 notebook**（除改 CFG 或引导骨架）。

**停止**：■ 中断本 cell。Kaggle 单次 9h。

**代码来源怎么认**：日志里有 `code source=hub|github|push-first` 与 `sha12`；
冷启动时 hub 还没导出 code.zip ⇒ `source=github` 是预期（不是 bug）。


In [ ]:
# @title Battle GPU Worker —— Push：先取代码再注册；Pull 填 hub_url
# 引导骨架（参数/保活/薄分支）留在本 cell；其余逻辑在 code.zip
# （remote/notebook_runtime.py + remote/push_bootstrap.py）。
import os, sys, threading, time

CFG = {
    "mode": "push",              # push | pull
    "hub_url": "",               # pull 必填；push 时可留空（registry_url 优先）
    "registry_url": "",          # 自动注册的 hub（留空 = 用 hub_url）
    "hub_token": "",             # 注册鉴权 + hub /code（和 push_token 不是同一把）
    "push_port": 8790,
    "push_token": "YOUR_TOKEN_HERE",   # 本节点 authKey（hub 用它调本机）
    "node_id": "",               # 留空 = gpu-<env>（按角色稳定，换 VM 不堆历史节点）
    "label": "gpu-colab",
    "repo_url": "https://github.com/HuangJian/battle.git",
    "branch": "goal-nn",
    "code_wait_sec": 600,        # 只作用于 push 的 /code 等待；pull 仍是 1h
    "allow_push_first": True,    # hub 与 GitHub 都不通时的最后兜底（手动/内联注册）
    "cloudflared_path": "",
    "device": "auto",
    "use_multi_gpu": True,
    "max_session_hours": 9,
    "poll_interval_sec": 5,
    "idle_floor_sec": 3600,
    "max_worker_restarts": 5,
}

def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] {msg}", flush=True)

_keepalive_stop = threading.Event()

def _keepalive_loop():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        _sentinel = "/tmp/battle-halt-request"
        while not _keepalive_stop.is_set():
            if os.path.exists(_sentinel):
                print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] [keepalive] 检测到停机哨兵 → 释放 Colab 实例", flush=True)
                try:
                    from google.colab import runtime
                    runtime.unassign()
                except Exception as _e:
                    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] [keepalive] unassign 失败: {_e}——请手工断开", flush=True)
                break
            try:
                from IPython.display import Javascript, display
                display(Javascript(
                    "function f(){document.querySelector('colab-connect-button')?.click();}"
                    "setTimeout(f,1000);"))
            except Exception:
                pass
            _keepalive_stop.wait(60)
    else:
        n = 0
        while not _keepalive_stop.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] [keepalive] alive ({n*2} min)", flush=True)
            n += 1
            _keepalive_stop.wait(120)

threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")

_hub = str(CFG.get("hub_url") or "").strip()
_mode = str(CFG.get("mode") or "pull").strip().lower()
_code_dir = "/tmp/worker-code"

# ── Pull：逐字节不变（1h 等 /code；401/403 直接 FATAL）──────────────────
if _mode == "pull":
    if not _hub:
        raise SystemExit("[battle-rl] FATAL: pull 需要 hub_url")
    import hashlib, io, urllib.error, urllib.request, zipfile
    from pathlib import Path
    _deadline = time.time() + 3600
    _attempt = 0
    while True:
        _attempt += 1
        try:
            _req = urllib.request.Request(
                _hub.rstrip("/") + "/code",
                headers={"Authorization": "Bearer " + str(CFG["hub_token"])})
            with urllib.request.urlopen(_req, timeout=120) as _resp:
                _raw = _resp.read()
            _log(f"code.zip 就绪: {len(_raw)} bytes sha={hashlib.sha256(_raw).hexdigest()[:12]}…")
            break
        except urllib.error.HTTPError as _e:
            if _e.code in (401, 403):
                raise SystemExit(f"[battle-rl] FATAL: /code HTTP {_e.code} — token 不一致") from None
            if _e.code != 404:
                _log(f"/code HTTP {_e.code} —— 30s 后重试")
            elif time.time() > _deadline:
                raise SystemExit("[battle-rl] FATAL: 等 code.zip 超 1h") from None
            time.sleep(30)
        except Exception as _e:
            _log(f"hub 连接异常（{type(_e).__name__}）—— 30s 后重试")
            time.sleep(30)
    Path(_code_dir).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(_raw)) as _z:
        _z.extractall(_code_dir)
    sys.path.insert(0, _code_dir)
    _log("移交 remote.notebook_runtime")
    from remote.notebook_runtime import run_notebook
    CFG["keepalive_stop"] = _keepalive_stop
    CFG["log"] = _log
    CFG["code_dir"] = _code_dir
    raise SystemExit(run_notebook(CFG))

# ── Push：代码优先（hub /code → GitHub clone），代码到位再起服务/注册 ─────
# 为什么不能 push-first 打头：没有 code.zip ⇒ 只有 bootstrap server；
# 没登记 ⇒ hub 不知道 URL ⇒ 不推 job ⇒ code.zip 永远不来（注册死锁）。
if _mode == "push":
    if CFG.get("push_token", "") in ("", "YOUR_TOKEN_HERE"):
        raise SystemExit("[battle-rl] FATAL: push_token 必填")
    import hashlib, io, json, shutil, subprocess, urllib.error, urllib.request, zipfile
    from pathlib import Path
    _code_base = str(CFG.get("registry_url") or _hub or "").strip().rstrip("/")
    if "your-tunnel" in _code_base:
        _code_base = ""  # 占位符/示例值不算配置
    _hub_tok = str(CFG.get("hub_token") or "").strip()
    _code_src = ""
    if _code_base and _hub_tok:
        _code_deadline = time.time() + float(CFG.get("code_wait_sec") or 600)
        _tries = 0
        while True:
            _tries += 1
            try:
                _req = urllib.request.Request(
                    _code_base + "/code", headers={"Authorization": "Bearer " + _hub_tok})
                with urllib.request.urlopen(_req, timeout=120) as _resp:
                    _raw = _resp.read()
                Path(_code_dir).mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(io.BytesIO(_raw)) as _z:
                    _z.extractall(_code_dir)
                _code_src = "hub"
                _log(f"code.zip 就绪（source=hub）: {len(_raw)} bytes sha={hashlib.sha256(_raw).hexdigest()[:12]}")
                break
            except urllib.error.HTTPError as _e:
                if _e.code in (401, 403):
                    _log(f"hub /code 鉴权失败 HTTP {_e.code} —— 转 GitHub clone（检查 hub_token）")
                    break
                if _e.code != 404:
                    _log(f"/code HTTP {_e.code} —— 重试（余 {int(max(0, _code_deadline - time.time()))}s）")
                elif _tries % 6 == 1:
                    _log("hub 还没导出 code.zip（404）——重试中…")
            except Exception as _e:
                _log(f"hub /code 异常（{type(_e).__name__}）——重试（余 {int(max(0, _code_deadline - time.time()))}s）")
            if time.time() > _code_deadline:
                _log(f"等 hub /code 超过 {int(CFG.get('code_wait_sec') or 600)}s —— 转 GitHub clone")
                break
            time.sleep(20)
    if not _code_src:
        _repo = str(CFG.get("repo_url") or "https://github.com/HuangJian/battle.git")
        _branch = str(CFG.get("branch") or "goal-nn")
        _tmp = Path(_code_dir + "-git")
        try:
            shutil.rmtree(_tmp, ignore_errors=True)
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", _branch, _repo, str(_tmp)],
                check=True, timeout=600)
            _sha = subprocess.run(
                ["git", "-C", str(_tmp), "rev-parse", "--short=12", "HEAD"],
                capture_output=True).stdout.decode("utf-8", "replace").strip()
            Path(_code_dir).mkdir(parents=True, exist_ok=True)
            shutil.copytree(_tmp, _code_dir, dirs_exist_ok=True)
            shutil.rmtree(_tmp, ignore_errors=True)
            _code_src = "github"
            _log(f"代码就绪（source=github @{_sha or '?'}，branch={_branch}）")
        except Exception as _e:
            _log(f"GitHub clone 失败（{type(_e).__name__}: {_e}）")

    if _code_src:
        sys.path.insert(0, _code_dir)
        _log(f"移交 remote.notebook_runtime（code source={_code_src}）")
        from remote.notebook_runtime import run_notebook
        CFG["keepalive_stop"] = _keepalive_stop
        CFG["log"] = _log
        CFG["code_dir"] = _code_dir
        raise SystemExit(run_notebook(CFG))

    if not CFG.get("allow_push_first", True):
        raise SystemExit(
            "[battle-rl] FATAL: push 启动需要完整代码；hub /code 与 GitHub 都不可达。"
            "检查网络/凭据，或改 mode='pull' 走任务包引导")

    _log("⚠ 两条代码来源都不可达 —— 退回 push-first（等首个带 code.zip 的 job；不推荐）")


def _bootstrap_push_first():
    """最后兜底：首包前没有代码，只能内联 HTTP + 隧道 + 内联注册。"""
    import base64, io, json, secrets, subprocess, zipfile
    from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
    from pathlib import Path
    from urllib.request import Request, urlopen

    _port = int(CFG["push_port"])
    _tok = str(CFG["push_token"])
    _work = Path("/tmp/remote-worker-serve"); _work.mkdir(parents=True, exist_ok=True)
    _cdir = Path(_code_dir)
    _upg = {"body": None}

    class _H(BaseHTTPRequestHandler):
        def log_message(self, *a): pass
        def _j(self, o, s=200):
            b = json.dumps(o).encode(); self.send_response(s)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
        def _ok(self):
            return secrets.compare_digest(self.headers.get("Authorization", ""), f"Bearer {_tok}")
        def do_GET(self):
            if not self._ok(): return self._j({"error": "unauthorized"}, 401)
            p = self.path.split("?", 1)[0]
            if p == "/ping":
                return self._j({"ok": True, "busy": False, "queued": 0, "done": 0, "bootstrap": True})
            if p == "/code-sha": return self._j({"cached": False})
            return self._j({"error": "nf"}, 404)
        def do_POST(self):
            if not self._ok(): return self._j({"error": "unauthorized"}, 401)
            if self.path.split("?", 1)[0] != "/job": return self._j({"error": "nf"}, 404)
            raw = self.rfile.read(int(self.headers.get("Content-Length", "0")))
            try: body = json.loads(raw.decode())
            except ValueError: return self._j({"error": "bad json"}, 400)
            if not body.get("code_b64"):
                return self._j({"error": "code-missing"}, 428)
            try:
                _cdir.mkdir(parents=True, exist_ok=True)
                zipfile.ZipFile(io.BytesIO(base64.b64decode(body["code_b64"]))).extractall(_cdir)
            except Exception as e:
                return self._j({"error": str(e)}, 400)
            _upg["body"] = raw
            _log(f"code.zip 已解包 -> {_cdir}（升级完整 worker_server）")
            return self._j({"status": "accepted", "upgrading": True}, 202)

    _srv = ThreadingHTTPServer(("0.0.0.0", _port), _H)
    threading.Thread(target=_srv.serve_forever, daemon=True).start()
    _log(f"push bootstrap :{_port}（无代码来源）")

    _cf = str(CFG.get("cloudflared_path") or "") or os.popen(
        "where cloudflared 2>nul || which cloudflared 2>/dev/null").read().strip()
    if not _cf:
        _log("cloudflared 安装…")
        _cf = "/usr/local/bin/cloudflared"
        subprocess.run(["curl", "-fsSL",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o", _cf], check=True, timeout=60)
        os.chmod(_cf, 0o755)
    _cf_log = _work / "cloudflared.log"
    _cfp = subprocess.Popen(
        [_cf, "tunnel", "--url", f"http://localhost:{_port}", "--logfile", str(_cf_log)],
        stdout=open(_cf_log, "w"), stderr=subprocess.STDOUT)
    _url, _t0 = None, time.time()
    while time.time() - _t0 < 60 and not _url:
        try:
            import re as _re
            _urls = _re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com",
                                Path(_cf_log).read_text(encoding="utf-8", errors="replace"))
            if _urls: _url = _urls[-1]; break
        except OSError:
            pass
        if _cfp.poll() is not None: break
        time.sleep(2)

    # 内联注册（没有代码 ⇒ 用不到 remote.push_register）：node_id 按角色稳定，
    # 写回 CFG 让升级后的 run_push_worker 复用同一个 id（否则 hub 多一台同 URL 的重复节点）。
    _env = ("colab" if ("COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ)
            else ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "local"))
    _node_id = str(CFG.get("node_id") or "") or f"gpu-{_env}"
    CFG["node_id"] = _node_id
    _reg_base = str(CFG.get("registry_url") or _hub or "").strip().rstrip("/")
    _hub_tok = str(CFG.get("hub_token") or "").strip()
    _snippet = json.dumps({"nodes": [{"id": _node_id, "url": _url or "", "authKey": _tok,
                                      "gpu_push": True, "enabled": True}]}, ensure_ascii=False)
    if _url:
        _log(f"★ 隧道 URL: {_url}")
    else:
        _log(f"⚠ 无隧道 URL——查 {_cf_log}")
    if _url and _reg_base and _hub_tok:
        try:
            _rq = Request(
                _reg_base + "/admin/nodes/register",
                data=json.dumps({"id": _node_id, "url": _url, "authKey": _tok,
                                 "kind": "gpu_push",
                                 "label": str(CFG.get("label") or "gpu-colab")}).encode(),
                headers={"Authorization": "Bearer " + _hub_tok,
                         "Content-Type": "application/json"}, method="POST")
            with urlopen(_rq, timeout=15) as _rs:
                _rb = json.loads(_rs.read().decode("utf-8") or "{}")
            if "capabilities" in _rb:
                _log(f"★ 已自动注册 id={_node_id} capabilities={_rb.get('capabilities')}")
            else:
                _log("WARN 老 hub 不回 capabilities —— 能力键可能没写进去；手工片段：")
                _log("  " + _snippet)
        except Exception as _e:
            _log(f"WARN 自动注册失败（{type(_e).__name__}）——手工片段：")
            _log("  " + _snippet)
    else:
        _log("自动注册未生效（缺 registry_url/hub_url 或 hub_token）——手工片段：")
        _log("  " + _snippet)

    _deadline = time.time() + int(CFG["max_session_hours"]) * 3600
    try:
        while _upg["body"] is None:
            if time.time() > _deadline:
                raise SystemExit(0)
            time.sleep(2)
    except KeyboardInterrupt:
        _log("收到中断"); raise SystemExit(0)
    finally:
        # 必须 server_close()：shutdown() 只停 serve_forever，**不释放监听套接字**。
        # 完整 worker_server 随后要 bind **同一个端口**（push-first 共用端口设计），
        # 不释放 ⇒ 子进程被 common/port_guard.py 拒（"端口已被占用"）⇒ 30s 超时
        # ⇒ SystemExit(-1) ⇒ cell 挂、隧道下线 ⇒ 控制台只看到「状态查询 530」。
        # 2026-09-15 实测事故。
        _srv.shutdown(); _srv.server_close(); time.sleep(0.5)

    # code 已在 _cdir —— 之后全部走 code.zip 运行时
    sys.path.insert(0, _code_dir)
    from remote.push_bootstrap import requeue_job, spawn_full_worker_server, wait_ping
    from remote.notebook_runtime import resolve_device, run_notebook

    # 设备必须在 spawn **之前**解析：spawn 传的是 --device，而 run_notebook 的
    # 解析更晚——不先解析就会把字面量 "auto" 交给 worker（torch.device("auto")
    # 当场炸掉整轮 job，2026-09-15 事故）。写好 device_resolved 后 run_notebook
    # 会复用它，探测只跑一次、日志只出一份。
    CFG["device_resolved"] = resolve_device(CFG, _log)
    _real = spawn_full_worker_server(
        _port, _tok, _work, str(CFG["device_resolved"]), _cdir, _work / "serve.log")
    if not wait_ping(_port, _tok, 30):
        # 真因在子进程 stdout/stderr 里（被重定向到 serve.log）——摊到 cell 输出，
        # 否则只剩一句「未就绪」，操作员得自己知道去 `!cat` 它（2026-09-15 实测踩过）。
        _log("worker_server 30s 未就绪——下面是 serve.log 尾部（真因通常在这）：")
        _slog = _work / "serve.log"
        try:
            _lines = _slog.read_text(encoding="utf-8", errors="replace").splitlines()
        except OSError as _e:
            _lines = [f"(无法读取 {_slog}：{_e})"]
        for _ln in _lines[-40:]:
            _log(f"  | {_ln}")
        _real.kill(); raise SystemExit(-1)
    _log("worker_server 就绪——重放首个 job 后守候")
    requeue_job(_port, _tok, _upg["body"] or b"{}")
    CFG["keepalive_stop"] = _keepalive_stop
    CFG["log"] = _log
    CFG["code_dir"] = _code_dir
    CFG["already_serving"] = {
        "serve_pid": _real.pid,
        "cf_pid": _cfp.pid if _cfp else None,
        "cf_url": _url,
    }
    raise SystemExit(run_notebook(CFG))


if _mode == "push":
    _bootstrap_push_first()
raise SystemExit(f"[battle-rl] FATAL: 未知 mode={_mode!r}（可选 'pull' / 'push'）")
